# Reward Model - Basic Example

This example demonstrates the setReward feature for defining custom
reward functions on a queueing network model and computing steady-state
expected rewards using the CTMC solver.

Reward functions allow modeling various performance metrics such as:
- Queue lengths
- Server utilization
- Blocking probabilities
- Custom cost functions

Copyright (c) 2012-2025, Imperial College London
All rights reserved.

In [ ]:
from line_solver import *
import math

In [ ]:
# Model Definition
# Create a simple M/M/1/K queue with finite buffer
model = Network('RewardExample')

In [ ]:
# Block 1: nodes
source = Source(model, 'Source')
queue = Queue(model, 'Queue', SchedStrategy.FCFS)
sink = Sink(model, 'Sink')

In [ ]:
# Set finite buffer capacity
queue.setNumberOfServers(1)
queue.setCapacity(3)  # Maximum jobs in the system

In [ ]:
# Block 2: job classes
oclass = OpenClass(model, 'Class1')
source.setArrival(oclass, Exp(2))  # Arrival rate = 2
queue.setService(oclass, Exp(3))   # Service rate = 3 (utilization ~ 0.67)

In [ ]:
# Block 3: topology
model.link(Network.serial_routing([source, queue, sink]))

In [ ]:
print('=== Reward Model - Basic Example ===\n')

In [ ]:
# Define Reward Functions
# setReward(name, rewardFn) where rewardFn uses state accessor methods

# Reward 1: Queue length (number of jobs in the queue)
model.setReward('QueueLength', lambda state: state.at(queue, oclass))

In [ ]:
# Reward 2: Utilization (1 if server busy, 0 if idle)
model.setReward('Utilization', Reward.utilization(queue, oclass))

In [ ]:
# Reward 3: Blocking indicator (1 if buffer full, 0 otherwise)
model.setReward('BlockingProb', Reward.blocking(queue))

In [ ]:
# Reward 4: Weighted queue cost (quadratic penalty for long queues)
model.setReward('QueueCost', lambda state: state.at(queue, oclass) ** 2)

In [ ]:
# Solve with CTMC Solver
print('Solving with CTMC solver...\n')

In [ ]:
options = {
    'verbose': 1
}

In [ ]:
solver = CTMC(model, options)

In [ ]:
# Get Steady-State Expected Rewards
R, names = solver.get_avg_reward()

In [ ]:
print('\n=== Steady-State Expected Rewards ===')
for i in range(len(names)):
    print(f'{names[i]:>15s}: {R[i]:.6f}')

In [ ]:
# Get Transient Reward Analysis: E[r(X(t))] over a finite horizon
tran_solver = CTMC(model, timespan=[0, 5], verbose=0)
Rt, t, names = tran_solver.get_tran_reward()

In [ ]:
print('\n=== Transient Reward Analysis ===')
print(f'Time horizon: [0, 5], time points: {len(t)}')
for r in Rt:
    print(f"{r['name']:>15s}: E[r(X(0))]={r['metric'][0]:.6f}  "
          f"E[r(X(T))]={r['metric'][-1]:.6f}")

In [ ]:
# Compare with Analytical Results for M/M/1/K
print('\n=== Comparison with M/M/1/K Analytical Results ===')

In [ ]:
lambda_rate = 2.0  # Arrival rate
mu = 3.0          # Service rate
rho = lambda_rate / mu
K = 10            # Buffer capacity

In [ ]:
# For M/M/1/K queue, steady-state probabilities:
# pi(n) = (1-rho) * rho^n / (1 - rho^(K+1))
if rho != 1:
    pi = []
    for n in range(K + 1):
        pi.append((1 - rho) * (rho ** n) / (1 - rho ** (K + 1)))
else:
    pi = [1.0 / (K + 1)] * (K + 1)

In [ ]:
# Analytical expected queue length
L_analytical = sum(n * pi[n] for n in range(K + 1))

In [ ]:
# Analytical utilization (P(server busy) = 1 - pi(0))
U_analytical = 1 - pi[0]

In [ ]:
# Analytical blocking probability = pi(K)
B_analytical = pi[K]

In [ ]:
# Analytical queue cost (E[N^2])
Cost_analytical = sum((n ** 2) * pi[n] for n in range(K + 1))

In [ ]:
print(f'{"QueueLength":>15s}: LINE = {R[0]:.6f}, Analytical = {L_analytical:.6f}, Error = {abs(R[0] - L_analytical):.2e}')
print(f'{"Utilization":>15s}: LINE = {R[1]:.6f}, Analytical = {U_analytical:.6f}, Error = {abs(R[1] - U_analytical):.2e}')
print(f'{"BlockingProb":>15s}: LINE = {R[2]:.6f}, Analytical = {B_analytical:.6f}, Error = {abs(R[2] - B_analytical):.2e}')
print(f'{"QueueCost":>15s}: LINE = {R[3]:.6f}, Analytical = {Cost_analytical:.6f}, Error = {abs(R[3] - Cost_analytical):.2e}')

In [ ]:
print('\nNote: Reward functions provide a powerful way to define custom performance')
print('      metrics beyond standard measures like utilization and queue length.')
print('      The CTMC solver can compute steady-state expected values and transient')
print('      behavior of these reward functions.')